PDF Ingestion Pipeline Vector Db Store

In [25]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
def process_all_pdfs_in_directory(directory_path):
    all_documents = []
    pdf_dir =  Path(directory_path)
    #Find all PDF files in the directory and its subdirectories
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory and its subdirectories.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        try: 
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source Metadata to each document
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"Processed {len(all_documents)} documents from PDF files.")
    return all_documents

all_pdf_documents = process_all_pdfs_in_directory("../../Datasets/pdf_files")

In [27]:
# Text Splitting into chunks
def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_documents = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_documents)} chunks.")

    if split_documents:
        print("\n Example of a chunk:")
        print(f"Content: {split_documents[0].page_content[:100]}...")
        print(f"Metadata: {split_documents[0].metadata}")
    return split_documents

In [28]:
chunks = split_documents_into_chunks(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Split 171 documents into 750 chunks.

 Example of a chunk:
Content: Language Models are Few-Shot Learners
Tom B. Brown∗
Benjamin Mann∗
Nick Ryder∗
Melanie Subbiah∗
Jare...
Metadata: {'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-07-24T00:04:08+00:00', 'source': '..\\..\\Datasets\\pdf_files\\gpt3_models.pdf', 'file_path': '..\\..\\Datasets\\pdf_files\\gpt3_models.pdf', 'total_pages': 75, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-07-24T00:04:08+00:00', 'trapped': '', 'modDate': 'D:20200724000408Z', 'creationDate': 'D:20200724000408Z', 'page': 0, 'source_file': 'gpt3_models.pdf', 'file_type': 'pdf'}


### Embedding and VectorDbStore

In [29]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import hashlib
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [30]:
class EmbeddingManager:
    def __init__(self, model_name = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loaded embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully. Model dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded. Please load the model before generating embeddings.")
        print(f"Generating embeddings for {len(texts)} texts using model {self.model_name}...")
        embeddings = self.model.encode(
            texts, batch_size=32, normalize_embeddings=True,
            convert_to_numpy=True, show_progress_bar=len(texts) > 1,
        )
        print(f"Generated embeddings for {len(texts)} texts. Embedding shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager(model_name='all-MiniLM-L6-v2')

Loaded embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1385.41it/s]


Model all-MiniLM-L6-v2 loaded successfully. Model dimensions: 384


C:\Users\webgr\AppData\Local\Temp\ipykernel_10888\457054161.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model {self.model_name} loaded successfully. Model dimensions: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [31]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents_cosine_v1", persist_directory: str = "../../Database/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Collection of PDF document embeddings",
                    "hnsw:space": "cosine"
                }
            )

            print(f"ChromaDB initialized with collection '{self.collection_name}' at '{self.persist_directory}'")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray, batch_size: int = 256):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        if batch_size <= 0:
            raise ValueError("batch_size must be positive.")

        # Respect Chroma's write limit while keeping temporary memory bounded.
        if hasattr(self.client, "get_max_batch_size"):
            batch_size = min(batch_size, self.client.get_max_batch_size())
        for start in range(0, len(documents), batch_size):
            records = {}
            for i in range(start, min(start + batch_size, len(documents))):
                doc = documents[i]
                metadata = dict(doc.metadata)
                metadata.update(doc_index=i, content_length=len(doc.page_content))
                identity = f"{metadata.get('source', '')}|{metadata.get('page', '')}|{doc.page_content}"
                doc_id = hashlib.sha256(identity.encode("utf-8")).hexdigest()
                records[doc_id] = (doc.page_content, metadata, embeddings[i].tolist())

            # Stable IDs make repeated ingestion update records instead of duplicating them.
            self.collection.upsert(
                ids=list(records),
                documents=[record[0] for record in records.values()],
                metadatas=[record[1] for record in records.values()],
                embeddings=[record[2] for record in records.values()],
            )
        print(f"Stored {len(documents)} chunks; collection contains {self.collection.count()} records.")

vector_store = VectorStore()
vector_store

ChromaDB initialized with collection 'pdf_documents_cosine_v1' at '../../Database/vector_store'
Existing documents in collection: 0


In [32]:
#Convert the Texts to Embeddings
texts = [doc.page_content for doc in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store the embeddings to the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 750 texts using model all-MiniLM-L6-v2...


Batches: 100%|██████████| 24/24 [01:58<00:00,  4.96s/it]


Generated embeddings for 750 texts. Embedding shape: (750, 384)
Stored 750 chunks; collection contains 750 records.


### Data Retrieval Pipeline

In [33]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.2) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        query = query.strip()
        if not query:
            raise ValueError("query must not be empty.")
        if top_k <= 0:
            raise ValueError("top_k must be positive.")
        if not -1 <= score_threshold <= 1:
            raise ValueError("score_threshold must be between -1 and 1.")
        count = self.vector_store.collection.count()
        if count == 0:
            print("Collection is empty. Run the document ingestion cell first.")
            return []
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=min(top_k, count),
                include=["documents", "metadatas", "distances"],
            )
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = max(-1.0, min(1.0, 1.0 - float(distance)))  # Cosine collection
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                if not retrieved_docs and distances:
                    best_score = 1.0 - min(distances)
                    print(f"Best cosine score: {best_score:.3f}; threshold: {score_threshold:.3f}")
            else:
                print("No documents retrieved from the vector store.")
            return retrieved_docs
        except Exception as e:
            print(f"Error querying the vector store: {e}")
            raise
rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

In [34]:
query = "What is BART?"
rag_retriever.retrieve(query)


Retrieving documents for query: 'What is BART?' with top_k=5 and score_threshold=0.2
Generating embeddings for 1 texts using model all-MiniLM-L6-v2...
Generated embeddings for 1 texts. Embedding shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'f1c6b166cfb548cc5599260bb955e46b2b2ad5f2d41a1be4e5aa3c5593dff913',
  'content': 'responses. ‘?’ indicates factually incorrect responses, * indicates partially correct responses.\nTask\nInput\nModel\nGeneration\nMS-\nMARCO\ndeﬁne middle\near\nBART\n?The middle ear is the part of the ear between the middle ear and the nose.\nRAG-T The middle ear is the portion of the ear internal to the eardrum.\nRAG-S The middle ear includes the tympanic cavity and the three ossicles.\nwhat currency\nneeded in\nscotland\nBART\nThe currency needed in Scotland is Pound sterling.\nRAG-T Pound is the currency needed in Scotland.\nRAG-S The currency needed in Scotland is the pound sterling.\nJeopardy\nQuestion\nGener\n-ation\nWashington\nBART\n?This state has the largest number of counties in the U.S.\nRAG-T It’s the only U.S. state named for a U.S. president\nRAG-S It’s the state where you’ll ﬁnd Mount Rainier National Park\nThe Divine\nComedy\nBART\n*This epic poem by Dante is divided into 3 parts